# Course 11 — Multi-Agent Coding Workflows & Coordination

This guided lab treats multi-agent development as a controlled delivery system. It uses a fictional Northstar Insurance change, deterministic rules, and synthetic evidence. Nothing here provisions an identity, grants access, executes a coding agent, approves a merge, or authorizes release.

## 1. Load the governed and unsafe workflows

The two packages describe the same six work units. The difference is coordination design: identity, leases, ownership, dependencies, independent evidence, and bounded integration.

In [ ]:
import copy
import importlib.util
import json
import sys
from pathlib import Path

module_spec = importlib.util.spec_from_file_location('course11_notebook_lab', Path('lab.py'))
assert module_spec and module_spec.loader
lab = importlib.util.module_from_spec(module_spec)
sys.modules[module_spec.name] = lab
module_spec.loader.exec_module(lab)

reference = lab.load_bundle()
candidate = lab.load_bundle(candidate=True)
reference_report = lab.review_workflow(reference)
candidate_report = lab.review_workflow(candidate)
assert reference_report['counts'] == {}
assert candidate_report['counts'] == {'review': 11, 'blocking': 50}
candidate_report['counts']

## 2. Inspect why the candidate must stop

A shared branch is only one symptom. The unsafe package also uses role labels instead of workload identities, permanent wildcard permissions, overlapping writers, missing leases and locks, unbounded delegation, weak verification independence, and broad integration authority.

In [ ]:
candidate_codes = sorted({item['code'] for item in candidate_report['findings']})
required_failures = {
    'SHARED_EXECUTION_BRANCH', 'AGENT_IDENTITY_UNVERIFIED',
    'PARALLEL_WRITE_COLLISION', 'DELEGATION_AUTHORITY_AMPLIFIED',
    'VERIFICATION_INDEPENDENCE_WEAK', 'INTEGRATION_AUTHORITY_TOO_BROAD'
}
assert required_failures <= set(candidate_codes)
lab.coordination_decision(candidate_report)

## 3. Derive the safe execution waves

Parallelism comes from the dependency graph, not the number of available agents. Contract work runs first; extraction and validation can then proceed concurrently because their writes are exclusive and their inputs are pinned.

In [ ]:
schedule = lab.topological_waves(reference['workflow'])
assert schedule['acyclic']
assert schedule['waves'][1] == ['AWU-BR-EXTRACTION', 'AWU-BR-VALIDATION']
schedule['waves']

## 4. Compare execution models without hiding cost

Synthetic lead-time units are useful only when review load, rework, integration failures, and forbidden attempts remain visible. The comparison is not a productivity benchmark.

In [ ]:
comparison = lab.compare_execution_models()
unsafe = next(item for item in comparison['models'] if item['name'] == 'unsafe_parallel')
governed = next(item for item in comparison['models'] if item['name'] == 'governed_multi_agent')
assert unsafe['lead_time_units'] < governed['lead_time_units']
assert unsafe['rework_units'] > governed['rework_units']
comparison

## 5. Pull work only when implementation and review capacity exist

A reviewer queue is part of system capacity. Starting work that cannot be reviewed increases work in progress and evidence staleness rather than throughput. A pull decision binds identity, permission, lease, and lock only after a unit is ready; planning does not reserve them.

In [ ]:
states = {item['id']: 'PROPOSED' for item in reference['workflow']['work_units']}
states['AWU-BR-CONTRACT'] = 'VERIFIED'
dispatch = lab.pull_dispatch(
    reference['workflow'], states, active_count=0,
    review_queue_depths={'ENGINEERING_REVIEW': 0, 'SECURITY_REVIEW': 0, 'DOMAIN_REVIEW': 1},
    review_capacities={'ENGINEERING_REVIEW': 2, 'SECURITY_REVIEW': 1, 'DOMAIN_REVIEW': 1},
)
assert dispatch['dispatchable'] == ['AWU-BR-EXTRACTION']
assert 'AWU-BR-VALIDATION' in dispatch['waiting_for_review_capacity']

snapshot_states = {item['id']: item['state'] for item in reference['workflow']['work_units']}
future_units = {item['id'] for item in reference['workflow']['work_units'] if item['state'] in {'PLANNED', 'READY'}}
assert future_units.isdisjoint({item['work_unit_id'] for item in reference['workflow']['assignments']})
conflict_assignment = {
    'id': 'ASSIGN-CONFLICT-020', 'work_unit_id': 'AWU-BR-CONFLICT', 'agent_identity': 'agent-conflict-01',
    'permissions': {'write_paths': ['src/underwriting/conflicts.py', 'tests/underwriting/test_conflicts.py'], 'lifecycle': 'assignment_bound_temporary', 'expires_at': '2026-10-01T17:00:00Z', 'self_provisioned': False},
    'lease': {'id': 'LEASE-CONFLICT-020', 'expires_at': '2026-10-01T17:00:00Z', 'heartbeat_policy': 'every_5_minutes', 'recovery_policy': 'inspect_before_reassign'},
}
conflict_lock = {'id': 'LOCK-CONFLICT', 'work_unit_id': 'AWU-BR-CONFLICT', 'assignment_id': 'ASSIGN-CONFLICT-020', 'lease_id': 'LEASE-CONFLICT-020', 'paths': ['src/underwriting/conflicts.py', 'tests/underwriting/test_conflicts.py']}
bound = lab.bind_dispatch_resources(reference['workflow'], snapshot_states, work_unit_id='AWU-BR-CONFLICT', assignment=conflict_assignment, lock=conflict_lock)
assert bound['outcome'] == 'ASSIGNED'
{'capacity_decision': dispatch, 'resource_binding': bound['outcome']}

## 6. Reconcile duplicate delivery idempotently

At-least-once delivery is normal in durable systems. Stable assignment IDs prevent a duplicate scheduler message from starting a second execution.

In [ ]:
assignment = reference['workflow']['assignments'][0]
first = lab.process_assignment_delivery({'assignments': {}}, assignment)
second = lab.process_assignment_delivery(first['state'], assignment)
assert first['started'] is True
assert second['started'] is False
second

## 7. Recover an expired lease; do not blindly resume

A lease bounds ownership in time. Expiry moves the unit to recovery, where the durable record and repository state must be inspected before reassignment.

In [ ]:
interrupted_assignment = copy.deepcopy(assignment)
interrupted_assignment['state'] = 'IN_PROGRESS'
interrupted_assignment['lease']['state'] = 'ACTIVE'
expired = lab.recover_expired_lease(interrupted_assignment, now='2030-01-01T00:00:00Z')
assert expired['expired'] is True
assert expired['next_state'] == 'RECOVERY_REQUIRED'
assert expired['may_reassign'] is True
expired

## 8. Invalidate integration evidence when a component changes

Verification is revision-bound. A changed component makes previous integration evidence stale even when the check once passed.

In [ ]:
integration = reference['integration']
changed_component = next(iter(integration['components']))
invalidation = lab.invalidate_integration_evidence(integration, changed_component, 'sha256:new')
assert invalidation['state'] == 'REVALIDATION_REQUIRED'
assert invalidation['stale_evidence_ids']
invalidation

## 9. Propagate a shared-contract change

A contract change is a coordination event. Active consumers pause; completed consumers and their evidence require impact-aware revalidation. Impact analysis does not approve the change.

In [ ]:
run_states = {item['id']: 'PROPOSED' for item in reference['workflow']['work_units']}
run_states.update({'AWU-BR-CONTRACT': 'VERIFIED', 'AWU-BR-EXTRACTION': 'IN_PROGRESS', 'AWU-BR-VALIDATION': 'COMPLETED'})
blast = lab.contract_change_blast_radius(
    reference['workflow'], 'ProposedUpdate', run_states,
    {'AWU-BR-VALIDATION': ['EVID-VALIDATION-01']}
)
assert blast['active'] == ['AWU-BR-EXTRACTION']
assert 'AWU-BR-VALIDATION' in blast['completed']
assert blast['evidence_invalidated'] == ['EVID-VALIDATION-01']
blast

## 10. Route failures and detect repair oscillation

Retries are policy decisions. Transient infrastructure failure can consume a bounded retry; policy denial stops; stale context must refresh; repeated A↔B repair signals a systemic problem.

In [ ]:
policy = reference['policy']
assert lab.classify_execution_failure('transient_infrastructure_failure', 1, policy)['action'] == 'BOUNDED_RETRY'
assert lab.classify_execution_failure('policy_denied', 1, policy)['action'] == 'STOP_AND_ROUTE'
history = [
    {'failure_class': 'test_failure', 'semantic_change': 'A_TO_B'},
    {'failure_class': 'test_failure', 'semantic_change': 'B_TO_A'},
    {'failure_class': 'test_failure', 'semantic_change': 'A_TO_B'},
]
analysis = lab.analyze_execution_history(history, policy)
assert analysis['repair_oscillation'] is True
assert analysis['action'] == 'SYSTEMIC_PROBLEM_SUSPECTED'
analysis

## 11. Evaluate the deterministic rules

The cases are transparent mutation fixtures, not representative production traffic. Exact results establish regression coverage for these rules only.

In [ ]:
evaluation = lab.evaluate_coordination_rules()
assert (evaluation['population'], evaluation['true_positive']) == (36, 38)
assert (evaluation['false_positive'], evaluation['false_negative']) == (0, 0)
assert evaluation['exact_matches'] == {'numerator': 36, 'denominator': 36}
evaluation['claim']

## 12. Production upgrade and learner experiments

Before production, replace JSON fixtures with authenticated workload identities, short-lived capability grants, protected branch/ruleset APIs, durable state and idempotency storage, typed verification/approval artifacts, immutable evidence, event tracing, reviewer-capacity telemetry, and independent assurance.

Try these experiments: saturate both review queues; remove a dependency; expire a different lease; amplify delegated authority; change a component after integration; or lower the retry budget. Predict the typed finding and recovery path before running the cell.